# Word2Vec і GloVe

## Що зробимо

1. Підготуємо невеликий корпус англомовних відгуків про продукт.
2. Очистимо текст, виконаємо токенізацію і перевіримо базову статистику корпусу.
3. Навчимо власну модель `Word2Vec` з параметрами із завдання.
4. Знайдемо семантично близькі слова для `good`, `bad`, `quality`.
5. Порівняємо власну модель із готовими векторами `GloVe`.
6. Повторимо аналіз для додаткових завдань: стоп-слова, `min_count=2`, 10 ключових слів і підсумкові висновки.


## Підготовка середовища

Потрібні бібліотеки: `pandas`, `scikit-learn`, `gensim`. Якщо `gensim` ще не встановлено, виконайте в окремій клітинці:

```python
%pip install gensim
```

Модель `GloVe` завантажується через `gensim.downloader`. Перше завантаження може тривати кілька хвилин, бо файл моделі великий.


In [1]:
from collections import Counter
import re
import random

import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from gensim.models import Word2Vec
import gensim.downloader as api

SEED = 42
random.seed(SEED)

pd.set_option("display.max_colwidth", 120)
print("Environment is ready")


Environment is ready


## 1. Підготовка корпусу текстів

Завдання вимагає 10-15 речень з відгуками або твітами про продукт. Візьмемо 15 коротких речень, де повторюються потрібні для аналізу слова: `good`, `bad`, `quality`, `service`, `price`, `delivery`, `support`, `product`, `app`, `packaging`.


In [2]:
reviews = [
    "The product has good quality and fast delivery.",
    "Good service and helpful support made the app easy to use.",
    "The price is fair and the product feels reliable.",
    "Bad packaging made the first delivery frustrating.",
    "The service was bad at first, but support solved the issue quickly.",
    "Great quality, good price, and simple setup instructions.",
    "Poor battery life makes the product hard to recommend.",
    "The delivery was quick, but the packaging looked cheap.",
    "I like the app design because it is clean and useful.",
    "Customer support gave good advice and friendly service.",
    "The product quality feels consistent after two weeks.",
    "The price is high, but the features add real value.",
    "Bad updates sometimes make the app slow and confusing.",
    "Fast delivery, reliable product, and great service.",
    "The packaging is nice, and the overall quality is good.",
]

reviews_df = pd.DataFrame({
    "review_id": range(1, len(reviews) + 1),
    "text": reviews,
})
reviews_df


,review_id,text
0,1,The product has good quality and fast delivery.
1,2,Good service and helpful support made the app easy to use.
2,3,The price is fair and the product feels reliable.
3,4,Bad packaging made the first delivery frustrating.
4,5,"The service was bad at first, but support solved the issue quickly."
5,6,"Great quality, good price, and simple setup instructions."
6,7,Poor battery life makes the product hard to recommend.
7,8,"The delivery was quick, but the packaging looked cheap."
8,9,I like the app design because it is clean and useful.
9,10,Customer support gave good advice and friendly service.


Очистимо текст: нижній регістр, видалення пунктуації, зайвих пробілів і токенізація. Результат збережемо як список списків, де кожен підсписок - токени одного речення.


In [3]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    return clean_text(text).split()


reviews_df["clean_text"] = reviews_df["text"].apply(clean_text)
reviews_df["tokens"] = reviews_df["text"].apply(tokenize)

tokenized_sentences = reviews_df["tokens"].tolist()
reviews_df[["review_id", "clean_text", "tokens"]]


,review_id,clean_text,tokens
0,1,the product has good quality and fast delivery,"[the, product, has, good, quality, and, fast, delivery]"
1,2,good service and helpful support made the app easy to use,"[good, service, and, helpful, support, made, the, app, easy, to, use]"
2,3,the price is fair and the product feels reliable,"[the, price, is, fair, and, the, product, feels, reliable]"
3,4,bad packaging made the first delivery frustrating,"[bad, packaging, made, the, first, delivery, frustrating]"
4,5,the service was bad at first but support solved the issue quickly,"[the, service, was, bad, at, first, but, support, solved, the, issue, quickly]"
5,6,great quality good price and simple setup instructions,"[great, quality, good, price, and, simple, setup, instructions]"
6,7,poor battery life makes the product hard to recommend,"[poor, battery, life, makes, the, product, hard, to, recommend]"
7,8,the delivery was quick but the packaging looked cheap,"[the, delivery, was, quick, but, the, packaging, looked, cheap]"
8,9,i like the app design because it is clean and useful,"[i, like, the, app, design, because, it, is, clean, and, useful]"
9,10,customer support gave good advice and friendly service,"[customer, support, gave, good, advice, and, friendly, service]"


In [4]:
sentence_count = len(tokenized_sentences)
avg_sentence_length = sum(len(sentence) for sentence in tokenized_sentences) / sentence_count

corpus_stats = pd.DataFrame({
    "metric": ["Кількість речень", "Середня довжина речення, слів"],
    "value": [sentence_count, round(avg_sentence_length, 2)],
})
corpus_stats


,metric,value
0,Кількість речень,15.00
1,"Середня довжина речення, слів",9.07


## 2. Модель Word2Vec

Навчаємо модель `Word2Vec` з параметрами із завдання: `vector_size=100`, `window=5`, `min_count=1`, `sg=1`. Для стабільнішого результату на малому корпусі збільшуємо кількість епох, а `workers=1` і `seed` роблять запуск відтворюванішим.


In [11]:
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,       # скільки сусідніх слів бачить модель
    min_count=1,    # мінімальна кількість появ слова у корпусі
    sg=1,           # вибір алгоритму
    seed=SEED,
    epochs=200,
)

print("Vocabulary size:", len(w2v_model.wv))
print("Vector size:", w2v_model.wv.vector_size)


Vocabulary size: 71
Vector size: 100


In [12]:
if "good" in w2v_model.wv:
    good_vector_preview = pd.DataFrame({
        "dimension": list(range(1, 11)),
        "value": w2v_model.wv["good"][:10].round(4),
    })
    display(good_vector_preview)
else:
    print("Слова 'good' немає в корпусі.")


,dimension,value
0,1,0.0949
1,2,0.1238
2,3,-0.0026
3,4,0.1167
4,5,-0.1025
5,6,0.1146
6,7,-0.1463
7,8,0.1819
8,9,-0.1681
9,10,0.0607


## 3. Семантична близькість слів

Метод `most_similar()` шукає слова з найбільшою косинусною схожістю до цільового слова. На такому малому корпусі результат треба сприймати як навчальну демонстрацію, а не як надійну бізнес-модель.


In [7]:
def most_similar_table(model, target_words, topn=5):
    rows = []
    for word in target_words:
        if word not in model.wv:
            rows.append({
                "Цільове слово": word,
                "Близькі слова": "немає в словнику",
                "Косинусна схожість": "-",
            })
            continue

        similar = model.wv.most_similar(word, topn=topn)
        rows.append({
            "Цільове слово": word,
            "Близькі слова": ", ".join(item for item, score in similar),
            "Косинусна схожість": ", ".join(f"{score:.2f}" for item, score in similar),
        })
    return pd.DataFrame(rows)


basic_similarity_df = most_similar_table(w2v_model, ["good", "bad", "quality"], topn=5)
basic_similarity_df


,Цільове слово,Близькі слова,Косинусна схожість
0,good,"and, quality, price, is, instructions","1.00, 1.00, 1.00, 1.00, 1.00"
1,bad,"delivery, solved, support, the, high","1.00, 1.00, 1.00, 1.00, 1.00"
2,quality,"good, hard, and, is, the","1.00, 1.00, 1.00, 1.00, 1.00"


Коротке тлумачення: модель бачить тільки наші 15 речень, тому вона добре ловить локальні співвживання, але може плутати позитивні й негативні слова, якщо вони часто стоять у схожих позиціях речення.


## 4. Порівняння Word2Vec і GloVe

Спочатку виберемо 20 найчастіших слів корпусу. Потім для 7 слів порівняємо найближчі слова у власній моделі `Word2Vec` і в готовій моделі `GloVe`.


In [8]:
word_counts = Counter(word for sentence in tokenized_sentences for word in sentence)
top_20_words_df = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])
top_20_words_df


,word,frequency
0,the,17
1,and,9
2,product,5
3,good,5
4,is,5
5,quality,4
6,delivery,4
7,service,4
8,support,3
9,app,3


In [9]:
GLOVE_MODEL_NAME = "glove-wiki-gigaword-100"

try:
    glove_vectors = api.load(GLOVE_MODEL_NAME)
    GLOVE_AVAILABLE = True
    print(f"Loaded {GLOVE_MODEL_NAME}. Vocabulary size: {len(glove_vectors):,}")
except Exception as exc:
    glove_vectors = None
    GLOVE_AVAILABLE = False
    print("GloVe model was not loaded.")
    print("Reason:", repr(exc))
    print("Run this notebook with internet access, or rerun the cell after the model is cached locally.")


Loaded glove-wiki-gigaword-100. Vocabulary size: 400,000


In [10]:
def nearest_from_w2v(model, word, topn=5):
    if word not in model.wv:
        return []
    return model.wv.most_similar(word, topn=topn)


def nearest_from_glove(glove_model, word, topn=5):
    if glove_model is None:
        return []
    if word not in glove_model.key_to_index:
        return []
    return glove_model.most_similar(word, topn=topn)


def format_neighbors(items):
    if not items:
        return "немає даних"
    return ", ".join(f"{word} ({score:.2f})" for word, score in items)


def compare_comment(w2v_items, glove_items):
    if not w2v_items or not glove_items:
        return "Порівняння потребує доступної моделі GloVe і слова в обох словниках."

    w2v_words = {word for word, score in w2v_items}
    glove_words = {word for word, score in glove_items}
    overlap = sorted(w2v_words & glove_words)

    if overlap:
        return "Є перетин: " + ", ".join(overlap) + ". Обидві моделі частково ловлять схожий контекст."
    return "Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."


comparison_words = ["good", "quality", "service", "price", "delivery", "support", "bad"]
comparison_rows = []

for word in comparison_words:
    w2v_items = nearest_from_w2v(w2v_model, word, topn=5)
    glove_items = nearest_from_glove(glove_vectors, word, topn=5)
    comparison_rows.append({
        "Слово": word,
        "Найближчі слова Word2Vec": format_neighbors(w2v_items),
        "Найближчі слова GloVe": format_neighbors(glove_items),
        "Коментар": compare_comment(w2v_items, glove_items),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,Слово,Найближчі слова Word2Vec,Найближчі слова GloVe,Коментар
0,good,"and (1.00), quality (1.00), price (1.00), is (1.00), instructions (1.00)","better (0.89), sure (0.83), really (0.83), kind (0.83), very (0.83)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
1,quality,"good (1.00), hard (1.00), and (1.00), is (1.00), the (1.00)","excellent (0.76), standards (0.73), performance (0.72), good (0.72), better (0.72)",Є перетин: good. Обидві моделі частково ловлять схожий контекст.
2,service,"support (1.00), helpful (1.00), packaging (1.00), first (1.00), to (1.00)","services (0.88), public (0.69), network (0.68), private (0.67), system (0.66)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
3,price,"value (1.00), the (1.00), add (1.00), features (1.00), good (1.00)","prices (0.86), value (0.76), drop (0.75), stock (0.75), market (0.74)",Є перетин: value. Обидві моделі частково ловлять схожий контекст.
4,delivery,"bad (1.00), support (1.00), made (1.00), the (1.00), and (1.00)","deliveries (0.71), supplies (0.59), supply (0.58), dropped (0.58), crude (0.58)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
5,support,"service (1.00), made (1.00), delivery (1.00), bad (1.00), first (1.00)","supported (0.83), backing (0.77), supporting (0.77), efforts (0.73), supports (0.73)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
6,bad,"delivery (1.00), solved (1.00), support (1.00), the (1.00), high (1.00)","worse (0.79), good (0.77), things (0.77), too (0.76), thing (0.76)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."


### Висновок до основних практичних завдань

У власній `Word2Vec`-моделі найближчі слова залежать від того, як саме слова зустрічаються в 15 реченнях. Тому модель часто повертає слова з одного локального сценарію: доставка, сервіс, пакування, якість продукту. `GloVe` навчена на значно більшому корпусі, тому її асоціації зазвичай стабільніші й ближчі до загального значення слова. Різниця особливо помітна для слів `good`, `bad`, `quality`, бо в малому корпусі вони мають мало контекстів.


## 5. Підготовка даних зі стоп-словами

Видалимо англомовні стоп-слова. Після цього перевіримо частотність, щоб упевнитися, що ключові слова залишилися в корпусі щонайменше двічі.


In [11]:
stop_words = set(ENGLISH_STOP_WORDS)


def remove_stop_words(tokens: list[str]) -> list[str]:
    return [token for token in tokens if token not in stop_words]


tokenized_no_stop = [remove_stop_words(tokens) for tokens in tokenized_sentences]

no_stop_counts = Counter(word for sentence in tokenized_no_stop for word in sentence)
no_stop_preview_df = pd.DataFrame(no_stop_counts.most_common(20), columns=["word", "frequency"])

pd.DataFrame({
    "sentence_id": range(1, len(tokenized_no_stop) + 1),
    "tokens_without_stop_words": tokenized_no_stop,
}).head(10)


,sentence_id,tokens_without_stop_words
0,1,"[product, good, quality, fast, delivery]"
1,2,"[good, service, helpful, support, app, easy, use]"
2,3,"[price, fair, product, feels, reliable]"
3,4,"[bad, packaging, delivery, frustrating]"
4,5,"[service, bad, support, solved, issue, quickly]"
5,6,"[great, quality, good, price, simple, setup, instructions]"
6,7,"[poor, battery, life, makes, product, hard, recommend]"
7,8,"[delivery, quick, packaging, looked, cheap]"
8,9,"[like, app, design, clean, useful]"
9,10,"[customer, support, gave, good, advice, friendly, service]"


In [12]:
no_stop_preview_df


,word,frequency
0,product,5
1,good,5
2,quality,4
3,delivery,4
4,service,4
5,support,3
6,app,3
7,price,3
8,bad,3
9,packaging,3


## 6. Word2Vec із `min_count=2` і 10 ключових слів

Через `min_count=2` у словнику залишаться лише слова, які трапляються мінімум двічі. Це зменшує шум, але на маленькому корпусі може викинути частину корисних слів.


In [13]:
w2v_min2_model = Word2Vec(
    sentences=tokenized_no_stop,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    seed=SEED,
    workers=1,
    epochs=200,
)

key_words = [
    "good", "service", "price", "delivery", "quality",
    "support", "product", "app", "packaging", "bad",
]

vocabulary_check_df = pd.DataFrame({
    "keyword": key_words,
    "frequency_after_stopword_removal": [no_stop_counts[word] for word in key_words],
    "in_word2vec_vocab": [word in w2v_min2_model.wv for word in key_words],
})

print("Vocabulary size with min_count=2:", len(w2v_min2_model.wv))
vocabulary_check_df


Vocabulary size with min_count=2: 14


,keyword,frequency_after_stopword_removal,in_word2vec_vocab
0,good,5,True
1,service,4,True
2,price,3,True
3,delivery,4,True
4,quality,4,True
5,support,3,True
6,product,5,True
7,app,3,True
8,packaging,3,True
9,bad,3,True


In [14]:
keyword_similarity_df = most_similar_table(w2v_min2_model, key_words, topn=5)
keyword_similarity_df


,Цільове слово,Близькі слова,Косинусна схожість
0,good,"app, product, service, feels, fast","0.21, 0.13, 0.12, 0.11, 0.07"
1,service,"app, delivery, great, fast, price","0.19, 0.19, 0.15, 0.14, 0.13"
2,price,"great, bad, delivery, service, fast","0.23, 0.18, 0.16, 0.13, 0.09"
3,delivery,"great, service, reliable, price, quality","0.23, 0.19, 0.18, 0.16, 0.11"
4,quality,"delivery, reliable, packaging, product, good","0.11, 0.07, 0.05, 0.01, -0.02"
5,support,"packaging, feels, reliable, app, fast","0.12, 0.07, 0.04, 0.00, -0.02"
6,product,"great, app, good, feels, service","0.21, 0.17, 0.13, 0.08, 0.05"
7,app,"good, service, product, great, packaging","0.21, 0.19, 0.17, 0.15, 0.05"
8,packaging,"support, fast, reliable, good, app","0.12, 0.11, 0.10, 0.06, 0.05"
9,bad,"great, fast, price, product, service","0.33, 0.24, 0.18, 0.04, 0.04"


## 7. Порівняння близьких слів у двох моделях

Для тих самих 10 ключових слів порівняємо локальні зв'язки власної `Word2Vec` і загальномовні зв'язки `GloVe`.


In [15]:
extended_rows = []

for word in key_words:
    w2v_items = nearest_from_w2v(w2v_min2_model, word, topn=5)
    glove_items = nearest_from_glove(glove_vectors, word, topn=5)
    extended_rows.append({
        "Ключове слово": word,
        "Word2Vec: близькі слова": format_neighbors(w2v_items),
        "GloVe: близькі слова": format_neighbors(glove_items),
        "Висновок": compare_comment(w2v_items, glove_items),
    })

extended_comparison_df = pd.DataFrame(extended_rows)
extended_comparison_df


,Ключове слово,Word2Vec: близькі слова,GloVe: близькі слова,Висновок
0,good,"app (0.21), product (0.13), service (0.12), feels (0.11), fast (0.07)","better (0.89), sure (0.83), really (0.83), kind (0.83), very (0.83)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
1,service,"app (0.19), delivery (0.19), great (0.15), fast (0.14), price (0.13)","services (0.88), public (0.69), network (0.68), private (0.67), system (0.66)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
2,price,"great (0.23), bad (0.18), delivery (0.16), service (0.13), fast (0.09)","prices (0.86), value (0.76), drop (0.75), stock (0.75), market (0.74)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
3,delivery,"great (0.23), service (0.19), reliable (0.18), price (0.16), quality (0.11)","deliveries (0.71), supplies (0.59), supply (0.58), dropped (0.58), crude (0.58)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
4,quality,"delivery (0.11), reliable (0.07), packaging (0.05), product (0.01), good (-0.02)","excellent (0.76), standards (0.73), performance (0.72), good (0.72), better (0.72)",Є перетин: good. Обидві моделі частково ловлять схожий контекст.
5,support,"packaging (0.12), feels (0.07), reliable (0.04), app (0.00), fast (-0.02)","supported (0.83), backing (0.77), supporting (0.77), efforts (0.73), supports (0.73)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
6,product,"great (0.21), app (0.17), good (0.13), feels (0.08), service (0.05)","products (0.82), sales (0.76), distribution (0.71), brand (0.70), marketing (0.70)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
7,app,"good (0.21), service (0.19), product (0.17), great (0.15), packaging (0.05)","iphone (0.75), ipad (0.74), ios (0.74), apps (0.74), android (0.72)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
8,packaging,"support (0.12), fast (0.11), reliable (0.10), good (0.06), app (0.05)","products (0.73), product (0.68), labels (0.63), recycling (0.62), manufacturing (0.62)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."
9,bad,"great (0.33), fast (0.24), price (0.18), product (0.04), service (0.04)","worse (0.79), good (0.77), things (0.77), too (0.76), thing (0.76)","Word2Vec відображає малий корпус відгуків, а GloVe дає ширші загальномовні асоціації."


## 8. Підсумкова таблиця і висновок

Нижче автоматично виділимо приклади, де результати двох моделей частково перетинаються, і приклади, де вони суттєво відрізняються.


In [16]:
def neighbor_words(items):
    return {word for word, score in items}

summary_rows = []

for word in key_words:
    w2v_items = nearest_from_w2v(w2v_min2_model, word, topn=5)
    glove_items = nearest_from_glove(glove_vectors, word, topn=5)
    overlap = sorted(neighbor_words(w2v_items) & neighbor_words(glove_items))

    if not glove_items:
        group = "GloVe недоступна"
        explanation = "Запустіть notebook з доступом до GloVe, щоб отримати повне порівняння."
    elif overlap:
        group = "схожі результати"
        explanation = "Є спільні близькі слова: " + ", ".join(overlap) + "."
    else:
        group = "суттєво відрізняються"
        explanation = "Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."

    summary_rows.append({
        "Ключове слово": word,
        "Тип прикладу": group,
        "Пояснення": explanation,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,Ключове слово,Тип прикладу,Пояснення
0,good,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
1,service,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
2,price,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
3,delivery,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
4,quality,схожі результати,Є спільні близькі слова: good.
5,support,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
6,product,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
7,app,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
8,packaging,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."
9,bad,суттєво відрізняються,"Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус."


In [17]:
similar_examples = summary_df[summary_df["Тип прикладу"] == "схожі результати"].head(3)
different_examples = summary_df[summary_df["Тип прикладу"] == "суттєво відрізняються"].head(3)

print("Приклади, де результати схожі:")
if len(similar_examples) == 0:
    print("- Немає явних перетинів у top-5 для цього малого корпусу.")
else:
    for _, row in similar_examples.iterrows():
        print(f"- {row['Ключове слово']}: {row['Пояснення']}")

print("\nПриклади, де результати суттєво відрізняються:")
if len(different_examples) == 0:
    print("- Немає прикладів для показу або GloVe недоступна.")
else:
    for _, row in different_examples.iterrows():
        print(f"- {row['Ключове слово']}: {row['Пояснення']}")


Приклади, де результати схожі:
- quality: Є спільні близькі слова: good.

Приклади, де результати суттєво відрізняються:
- good: Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус.
- service: Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус.
- price: Мала Word2Vec спирається на локальні речення, GloVe - на великий попередній корпус.


### Висновок

Розмір корпусу прямо впливає на якість semantic similarity. На 15 реченнях `Word2Vec` бачить лише локальні співвживання, тому результати можуть бути нестабільними: слова стають близькими не через загальне значення, а через випадково схожий контекст у малому наборі даних. `GloVe` працює стабільніше, бо навчена на великому корпусі й має багатший словник. Власну `Word2Vec` варто використовувати, коли є достатньо багато доменних текстів; готові embeddings краще підходять як сильна базова модель для малих корпусів.
